In [ ]:
import os
import numpy as np
import xarray as xr
import xesmf as xe

# -----------------------------
# 1) Open dataset and detect mask variable
# -----------------------------
fn = "/wolke_scratch/dnikolo/ICON_Data/Nudged_run_cph_13_02/ryan_nudged_test_ctp_2d_ml_20080903T000000Z.nc"
ds = xr.open_dataset(fn)

# Handle possible typo clmast vs ctmask
mask_name = "ctmask" if "ctmask" in ds.data_vars else ("clmast" if "clmast" in ds.data_vars else None)
if mask_name is None:
    raise ValueError("No mask variable found. Expected 'ctmask' (or typo 'clmast').")

# Ensure required vars exist
for v in ["ctp", "ctt"]:
    if v not in ds.data_vars:
        raise ValueError(f"Variable '{v}' not found in dataset.")

# -----------------------------
# 2) Build source lon/lat in degrees
#    Try common ICON names first
# -----------------------------
# Common possibilities
lon_candidates = ["clon", "lon", "longitude"]
lat_candidates = ["clat", "lat", "latitude"]

src_lon_name = next((v for v in lon_candidates if v in ds.variables), None)
src_lat_name = next((v for v in lat_candidates if v in ds.variables), None)

if src_lon_name is None or src_lat_name is None:
    raise ValueError(
        "Could not find source lon/lat coordinates. "
        "Expected one of clon/lon/longitude and clat/lat/latitude."
    )

src_lon = ds[src_lon_name]
src_lat = ds[src_lat_name]

# Convert radians -> degrees if needed (ICON clon/clat are often radians)
def to_degrees_if_radians(da, name):
    vals = da.values
    # Heuristic: radians likely if abs(max) <= ~2*pi
    vmax = np.nanmax(np.abs(vals))
    if vmax <= 6.5:
        print(f"Converting {name} from radians to degrees.")
        return xr.apply_ufunc(np.rad2deg, da)
    else:
        print(f"{name} appears to be in degrees; no conversion.")
        return da

src_lon_deg = to_degrees_if_radians(src_lon, src_lon_name)
src_lat_deg = to_degrees_if_radians(src_lat, src_lat_name)

# Normalize longitude to [-180, 180] (optional but often helpful)
src_lon_deg = ((src_lon_deg + 180) % 360) - 180

# -----------------------------
# 3) Prepare source grid dataset for xESMF
#    For unstructured source, xESMF expects 1D lon/lat + mask
# -----------------------------
# Determine source horizontal dimension from mask
# e.g., "cell", "ncells", etc.
mask_da = ds[mask_name]
if mask_da.ndim != 1:
    # If mask has time or extra dims, take first slice sensibly
    # You can adapt this if needed.
    non_spatial = [d for d in mask_da.dims if d not in src_lon_deg.dims and d not in src_lat_deg.dims]
    if non_spatial:
        sel_dict = {d: 0 for d in non_spatial}
        mask_da = mask_da.isel(sel_dict)
    # squeeze remaining singleton dims
    mask_da = mask_da.squeeze(drop=True)

if mask_da.ndim != 1:
    raise ValueError(
        f"Mask '{mask_name}' must resolve to 1D over source cells. Got dims={mask_da.dims}."
    )

src_dim = mask_da.dims[0]

# Align lon/lat to same dim/order
# If lon/lat are not explicitly on src_dim, try to squeeze/rename
def align_to_src_dim(coord, src_dim):
    c = coord.squeeze(drop=True)
    if c.ndim != 1:
        raise ValueError(f"Coordinate {coord.name} is not 1D after squeeze; dims={coord.dims}")
    if c.dims[0] != src_dim:
        c = c.rename({c.dims[0]: src_dim})
    return c

src_lon_deg = align_to_src_dim(src_lon_deg, src_dim)
src_lat_deg = align_to_src_dim(src_lat_deg, src_dim)

# Build source mask in ESMF convention: 1=valid, 0=masked
src_mask = xr.where(mask_da > 0.5, 1, 0).astype("int32")

ds_in = xr.Dataset(
    data_vars={"mask": (src_dim, src_mask.data)},
    coords={
        "lon": (src_dim, src_lon_deg.data),
        "lat": (src_dim, src_lat_deg.data),
    },
)

# -----------------------------
# 4) Build destination regular lat-lon grid
# -----------------------------
# Choose resolution:
dlat = 0.25
dlon = 0.25

lat_out = np.arange(-90 + dlat / 2, 90, dlat)      # cell centers
lon_out = np.arange(-180 + dlon / 2, 180, dlon)    # cell centers

ds_out = xr.Dataset(coords={"lat": lat_out, "lon": lon_out})

# Optional destination mask (example): keep all cells valid by default
# ds_out["mask"] = xr.DataArray(np.ones((lat_out.size, lon_out.size), dtype="int32"), dims=("lat", "lon"))

# -----------------------------
# 5) Create regridders
# -----------------------------
# For masked geophysical fields, conservative_normed is often robust near coasts/masks.
# For unstructured->regular conservative remap, xESMF/ESMF may require cell bounds on some grids.
# If conservative fails due to missing bounds, use nearest_s2d or bilinear as fallback.

weights_dir = "weights"
os.makedirs(weights_dir, exist_ok=True)

regridder_method_preference = ["conservative_normed", "bilinear", "nearest_s2d"]
regridders = {}

for method in regridder_method_preference:
    wfile = os.path.join(weights_dir, f"icon_to_ll_{method}_{dlat}deg.nc")
    try:
        rg = xe.Regridder(
            ds_in,
            ds_out,
            method=method,
            filename=wfile,
            reuse_weights=os.path.exists(wfile),
            unmapped_to_nan=True,  # important for masked/uncovered targets
        )
        regridders[method] = rg
        print(f"Built regridder: {method}")
    except Exception as e:
        print(f"Could not build {method}: {e}")

if not regridders:
    raise RuntimeError("No regridder could be created. Check grid/bounds compatibility.")

# Pick best available method by preference
for m in regridder_method_preference:
    if m in regridders:
        method_used = m
        regridder = regridders[m]
        break

print(f"Using method: {method_used}")

# -----------------------------
# 6) Apply mask BEFORE remapping data
# -----------------------------
ctp_in = ds["ctp"]
ctt_in = ds["ctt"]

# Align variable dims: ensure source cell dim name matches src_dim
def align_var(v, src_dim):
    # if src_dim not in dims, try rename a single 1D horizontal dim
    if src_dim not in v.dims:
        # heuristically pick dim that matches source size
        candidates = [d for d in v.dims if v.sizes[d] == ds_in.sizes[src_dim]]
        if len(candidates) == 1:
            v = v.rename({candidates[0]: src_dim})
        elif len(candidates) > 1:
            raise ValueError(f"Ambiguous source dim for {v.name}: {candidates}")
        else:
            raise ValueError(f"Could not find source horizontal dim in {v.name}. dims={v.dims}")
    return v

ctp_in = align_var(ctp_in, src_dim)
ctt_in = align_var(ctt_in, src_dim)

# Apply mask on source cells (invalid -> NaN)
valid_bool = src_mask.astype(bool)
ctp_masked = ctp_in.where(valid_bool)
ctt_masked = ctt_in.where(valid_bool)

# -----------------------------
# 7) Remap data fields
# -----------------------------
# xESMF expects dims with lon/lat info from ds_in; for unstructured grids,
# passing DataArray with src_dim works when regridder was built from ds_in.
ctp_out = regridder(ctp_masked)
ctt_out = regridder(ctt_masked)

# Also remap the mask itself for diagnostics
mask_frac_out = regridder(src_mask.astype("float32"))
# "strict" valid target mask threshold (choose threshold to your use case)
# e.g., require at least 50% source support
valid_out = mask_frac_out >= 0.5

# Enforce invalid target cells as NaN
ctp_out = ctp_out.where(valid_out)
ctt_out = ctt_out.where(valid_out)

# -----------------------------
# 8) Save outputs
# -----------------------------
out = xr.Dataset(
    data_vars={
        "ctp": ctp_out,
        "ctt": ctt_out,
        "ctmask_remap_fraction": mask_frac_out,
        "ctmask_target_valid": valid_out.astype("int8"),
    },
    coords={"lat": lat_out, "lon": lon_out},
    attrs={
        "source_file": fn,
        "regridding_method": method_used,
        "note": "ctmask_remap_fraction is remapped source-valid fraction; ctmask_target_valid uses threshold >=0.5",
    },
)

out_fn = "/wolke_scratch/dnikolo/ICON_Data/Nudged_run_cph_13_02/xESMF_regrid.nc"
out.to_netcdf(out_fn)
print(f"Saved: {out_fn}")